# EasyVitessce Example: Select cells interactively, then use the list of cell IDs

## Import EasyVitessce

This import statement enables interactive plots by default.
Refer to the [EasyVitessce documentation](https://vitessce.github.io/easy_vitessce/) for how to disable interactive plotting or configure other behaviors.

In [1]:
import easy_vitessce as ev

/Users/ericmoerth/ws/easy_vitessce/.venv/lib/python3.13/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/Users/ericmoerth/ws/easy_vitessce/.venv/lib/python3.13/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/Users/ericmoerth/ws/easy_vitessce/.venv/lib/python3.13/site-packages/spatialdata/_core/query/relational_query.py:532: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it 

## Dependencies for datasets

In [2]:
import scanpy as sc

## Load the example AnnData object

In [3]:
adata = sc.datasets.pbmc68k_reduced()

## Plot the data

Store the return value of `sc.pl.plotting_function()` in a variable.
See more details at https://vitessce.github.io/easy_vitessce/advanced.html#access-the-vitessce-configuration

In [4]:
vw = sc.pl.umap(adata, color="bulk_labels")
vw

VitessceWidget(js_dev_mode=True, uid='759a')

## Get the set of lasso-ed cell IDs

First, lasso a set of cells in the user interface.

See more details at https://vitessce.github.io/easy_vitessce/advanced.html#access-values-from-the-coordination-space

In [11]:
current_config = vw._config
obs_set_selection = current_config["coordinationSpace"]["obsSetSelection"]
additional_obs_sets = current_config["coordinationSpace"]["additionalObsSets"]

The value of `featureSelection[coordinationScope]` will be an array (to allow for multi-selection, which is not used here but is important for other plot types such as the dot plot).

In [12]:
obs_set_selection["A"]

[['My Selections', 'Selection 1']]

In [13]:
cell_ids = []
first_selected_set_path = obs_set_selection["A"][0]
first_selected_set_path_group_name = first_selected_set_path[0] # e.g., "My Selections"
first_selected_set_path_set_name = first_selected_set_path[1] # e.g., "Selection 1"

# The additional_obs_sets is technically a tree data structure,
# so we need to traverse it.
for group_node in additional_obs_sets["A"]["tree"]:
    if group_node["name"] == first_selected_set_path_group_name:
        for set_node in group_node["children"]:
            if set_node["name"] == first_selected_set_path_set_name:
                # The items in set_node["set"] are tuples like
                # (cell_id, prediction_score) but only care about the cell_id values here.
                cell_ids = [arr[0] for arr in set_node["set"]]

In [14]:
len(cell_ids)

158

### Subset the AnnData object to include only the selected cells

In [15]:
selection_adata = adata[cell_ids, :].copy()

#### Plot the filtered AnnData object

In [16]:
sc.pl.umap(selection_adata, color="bulk_labels")

VitessceWidget(js_dev_mode=True, uid='4695')